Impporting the dataset and creating X(features) and y(label)

In [ ]:
import pandas as pd
df=pd.read_csv("data.csv")
df


,ID,air_time1,disp_index1,gmrt_in_air1,gmrt_on_paper1,max_x_extension1,max_y_extension1,mean_acc_in_air1,mean_acc_on_paper1,mean_gmrt1,...,mean_jerk_in_air25,mean_jerk_on_paper25,mean_speed_in_air25,mean_speed_on_paper25,num_of_pendown25,paper_time25,pressure_mean25,pressure_var25,total_time25,class
0,id_1,5160,0.000013,120.804174,86.853334,957,6601,0.361800,0.217459,103.828754,...,0.141434,0.024471,5.596487,3.184589,71,40120,1749.278166,296102.7676,144605,P
1,id_2,51980,0.000016,115.318238,83.448681,1694,6998,0.272513,0.144880,99.383459,...,0.049663,0.018368,1.665973,0.950249,129,126700,1504.768272,278744.2850,298640,P
2,id_3,2600,0.000010,229.933997,172.761858,2333,5802,0.387020,0.181342,201.347928,...,0.178194,0.017174,4.000781,2.392521,74,45480,1431.443492,144411.7055,79025,P
3,id_4,2130,0.000010,369.403342,183.193104,1756,8159,0.556879,0.164502,276.298223,...,0.113905,0.019860,4.206746,1.613522,123,67945,1465.843329,230184.7154,181220,P
4,id_5,2310,0.000007,257.997131,111.275889,987,4732,0.266077,0.145104,184.636510,...,0.121782,0.020872,3.319036,1.680629,92,37285,1841.702561,158290.0255,72575,P
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,id_170,2930,0.000010,241.736477,176.115957,1839,6439,0.253347,0.174663,208.926217,...,0.119152,0.020909,4.508709,2.233198,96,44545,1798.923336,247448.3108,80335,H
170,id_171,2140,0.000009,274.728964,234.495802,2053,8487,0.225537,0.174920,254.612383,...,0.174495,0.017640,4.685573,2.806888,84,37560,1725.619941,160664.6464,345835,H
171,id_172,3830,0.000008,151.536989,171.104693,1287,7352,0.165480,0.161058,161.320841,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H
172,id_173,1760,0.000008,289.518195,196.411138,1674,6946,0.518937,0.202613,242.964666,...,0.114472,0.017194,3.493815,2.510601,88,51675,1915.573488,128727.1241,83445,H


In [ ]:
import numpy as np
features_column=df.columns[1:-1] #except id and label
label_column=df.columns[-1] #only label
X=df[features_column].values #returns only the values of the column
y_text=df[label_column].values
y=(y_text=='P').astype(int) #1-P and 0-H

Pre-processing

In [ ]:
from sklearn.preprocessing import StandardScaler #scaling
from sklearn.decomposition import PCA #reduce the dimension
from sklearn.pipeline import Pipeline #chain steps
preprocess=Pipeline([('scaler1',StandardScaler()),('pca',PCA(n_components=24,random_state=42)),('scaler2',StandardScaler())])
X_processing=preprocess.fit_transform(X) #fit and transform

Feed the processed input to quantum feature map via bandwidth scaled PQC. 8-qubit PQC with 24 parameters and a bandwidth. each row will be mapped to 24 rotation angles,scaled by a bandwidth b

In [ ]:
from qiskit import QuantumCircuit #to build circuits
from qiskit.circuit import ParameterVector
import numpy as np
"""
8-qubit ansatz with 24 parameters, bandwidth-scaled.

Structure: 3 encoding layers of RX→RY→RX alternating with 2 entangling layers of CZ→CX.
Total parameters: 24 (scaled to 8 qubits from 6-qubit's 24 params).
"""
def make_8q_pqc(bandwidth): #x is preprocessed, bandwidth scales all rotation angles
    x=ParameterVector("x",24)
    theta=bandwidth*x #scale the features
    qc=QuantumCircuit(8)#create a 8-qubit circuit
    idx=0
    #encoding layer 1:RX (8 params)
    for q in range(8):
        qc.rx(theta[idx],q)
        idx+=1
    #encoding layer 2:RY(8 params)
    for q in range(8):
        qc.ry(theta[idx],q)
        idx+=1
    #entangling layer 1:CZ ring
    for q in range(7):
        qc.cz(q,q+1)
    qc.cz(7,0) #close the ring
    #encoding layer 3 : RX (8 params)
    for q in range(8):
        qc.rx(theta[idx],q)
        idx+=1
    #entangling layer 2:CX ring
    for q in range(7):
        qc.cx(q,q+1)
    qc.cx(7,0)
    return qc



wrap the 8-qubit PQC into a quantum kernel+SVC and run on 60/20/20 splits


In [ ]:
from qiskit.primitives import StatevectorSampler
from qiskit import QuantumCircuit
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from sklearn.model_selection import ShuffleSplit,train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
import numpy as np
pqc=QuantumCircuit()
#let fidelity kernel use the internal default statevector simulator (noiseless simulator)
def make_8q_kernel(bandwidth): #FidelityQuantumKernel using 8q pqc and aer based estimator
    pqc=make_8q_pqc(bandwidth)
    qkernel=FidelityQuantumKernel(feature_map=pqc)
    return qkernel


In [ ]:
outer_cv=ShuffleSplit(n_splits=20,test_size=0.2,random_state=42)
bandwidth_grid=np.linspace(0.1,2.0,20)
test_accs_8q=[]
for i,(trainval_idx,test_idx) in enumerate(outer_cv.split(X_processing,y)):
    X_trainval,X_test=X_processing[trainval_idx],X_processing[test_idx]
    y_trainval,y_test=y[trainval_idx],y[test_idx]
    X_train,X_val,y_train,y_val=train_test_split(X_trainval,y_trainval,test_size=0.25,stratify=y_trainval,random_state=42)
    best_bw=None
    best_val_acc=-np.inf
    #bandwidth tuning on validation
    for bw in bandwidth_grid:
        qkernel=make_8q_kernel(bw)
        K_train=qkernel.evaluate(X_train,X_train)
        K_val=qkernel.evaluate(X_val,X_train)
        clf=SVC(kernel="precomputed",C=1.0,gamma="scale",tol=1e-3,random_state=42)
        clf.fit(K_train,y_train)
        val_acc=accuracy_score(y_val,clf.predict(K_val))
        if val_acc>best_val_acc:
            best_val_acc=val_acc
            best_bw=bw
    #retrain on train+val with best bandwidth and evaluate on test
    qkernel_best=make_8q_kernel(best_bw)
    K_train_full=qkernel_best.evaluate(X_trainval,X_trainval)
    K_test=qkernel_best.evaluate(X_test,X_trainval)
    clf_final=SVC(kernel="precomputed",C=1.0,gamma="scale",tol=1e-3,random_state=42)
    clf_final.fit(K_train_full,y_trainval)
    test_acc=accuracy_score(y_test,clf_final.predict(K_test))
    test_accs_8q.append(test_acc)
    print(f"split{i+1}:bw={best_bw:.2f},test_acc={test_acc*100:.2f}%")
mean_acc_8q=np.mean(test_accs_8q)*100
std_acc_8q=np.std(test_accs_8q)*100
print(f"\n8q mean accuracy: {mean_acc_8q:.2f} ± {std_acc_8q:.2f}")

split1:bw=0.80,test_acc=82.86%
split2:bw=0.30,test_acc=77.14%
split3:bw=0.30,test_acc=85.71%
split4:bw=0.20,test_acc=91.43%
split5:bw=0.40,test_acc=91.43%
split6:bw=0.30,test_acc=88.57%
split7:bw=0.30,test_acc=85.71%
split8:bw=0.40,test_acc=88.57%
split9:bw=0.10,test_acc=82.86%
split10:bw=0.30,test_acc=80.00%
split11:bw=0.80,test_acc=85.71%
split12:bw=0.30,test_acc=82.86%
split13:bw=0.20,test_acc=82.86%
split14:bw=0.30,test_acc=82.86%
split15:bw=0.20,test_acc=77.14%
split16:bw=0.30,test_acc=85.71%
split17:bw=0.40,test_acc=91.43%
split18:bw=0.80,test_acc=82.86%
split19:bw=0.20,test_acc=82.86%
split20:bw=0.30,test_acc=97.14%

8q mean accuracy: 85.29 ± 4.89
